In [1]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [79]:
df = pd.read_csv('Datasets/Student_Performance.csv')
#use below statement to convert True/False to 1/0
#df['Extracurricular Activities'] = df['Extracurricular Activities'].astype(int)
df.replace({'No': 0, 'Yes': 1}, inplace=True)

X = df.iloc[:,0:-1].values
y = df.iloc[:,-1].values

C:\Users\prajw\AppData\Local\Temp\ipykernel_7016\1009540167.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace({'No': 0, 'Yes': 1}, inplace=True)


In [80]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [6]:
#do basic data cleaning like: handling missing and null values

In [76]:
regression = LinearRegression()

In [81]:
regression.fit(X_train, y_train)

LinearRegression()

In [82]:
y_pred = regression.predict(X_test)

In [84]:
loss = ((y_pred - y_test)**2).mean()
print("Loss using OLS method is", loss)
print("Gradients are", regression.coef_, regression.intercept_)

Loss using OLS method is 4.133501329217882
Gradients are [2.85089934 1.01817231 0.62262256 0.48280056 0.19275124] -34.04704694801888


In [64]:
#build a neural network for this 1 dependent and 5 independent variables

In [42]:
X = torch.tensor(X.astype(np.float32))
y = torch.tensor(y, dtype=torch.float32).reshape(-1,1)
#above reshaping of tensor y is to further match the shape with y_pred(7000,1) to esaily calculate the loss

In [43]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [28]:
class neuralNetwork(nn.Module):
    def __init__(self):
        super(neuralNetwork, self).__init__()
        self.m = nn.Parameter(torch.zeros(5, 1))
        self.c = nn.Parameter(torch.zeros(1,1))

    def forward(self, x):
        y = (x@self.m + self.c)
        return y

        # earlier I returned torch.round(y). as a result, over the iterations, my gradients were not updating and stayed 0.
        # later, determined that torch.round(y) is not differentiable.

In [66]:
regression = neuralNetwork()
optimizer = torch.optim.SGD(regression.parameters(), lr=0.0001)
epoch = 1000

In [67]:
for i in range(epoch):
    y_pred = regression(X_train)
    loss = ((y_pred - y_train)**2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (i%50 == 0):
        print(i, loss, regression.m.grad, regression.c.grad)

0 tensor(3410.8159, grad_fn=<MeanBackward0>) tensor([[ -589.6510],
        [-8257.0020],
        [  -55.5049],
        [ -723.7932],
        [ -509.1218]]) tensor([[-110.3174]])
50 tensor(69.1434, grad_fn=<MeanBackward0>) tensor([[-26.1778],
        [  0.5271],
        [  0.6375],
        [ 10.0071],
        [  5.3300]]) tensor([[1.9408]])
100 tensor(65.2986, grad_fn=<MeanBackward0>) tensor([[-24.3753],
        [  0.4381],
        [  0.6461],
        [  9.8548],
        [  5.0177]]) tensor([[1.9613]])
150 tensor(61.9089, grad_fn=<MeanBackward0>) tensor([[-22.7074],
        [  0.3585],
        [  0.6531],
        [  9.6837],
        [  4.7134]]) tensor([[1.9782]])
200 tensor(58.9157, grad_fn=<MeanBackward0>) tensor([[-21.1634],
        [  0.2878],
        [  0.6587],
        [  9.4973],
        [  4.4183]]) tensor([[1.9919]])
250 tensor(56.2689, grad_fn=<MeanBackward0>) tensor([[-19.7334],
        [  0.2263],
        [  0.6630],
        [  9.2982],
        [  4.1334]]) tensor([[2.0029]]

In [68]:
#calculate the loss on test data

In [69]:
y_pred = regression(X_test)
loss = ((y_pred - y_test)**2).mean()
print("Loss in Neural network is:",loss)
print("Gradients are:",regression.m.grad, regression.c.grad)

Loss in Neural network is: tensor(38.1686, grad_fn=<MeanBackward0>)
Gradients are: tensor([[-7.2924],
        [-0.1396],
        [ 0.6474],
        [ 5.9423],
        [ 1.2090]]) tensor([[1.9926]])
